<a href="https://colab.research.google.com/github/ianjpdepaul-rgb/Coupled-Manifold/blob/main/SharingAToy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Licensed under the Apache License, Version 2.0
**Copyright 2026. Ian J. Preston-Campbell.**

*Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at [http://apache.org](http://apache.org)**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/coupled_manifold_live', exist_ok=True)

In [ ]:
!pip install -q transformers accelerate sentencepiece huggingface_hub gradio

In [ ]:
from huggingface_hub import login
import os
login(token=os.environ.get("HF_TOKEN"))

In [4]:
"""
COUPLED MANIFOLD — Live Reasoning Session (Full Ongoing Suite)
Model: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B

BEFORE THIS CELL:
  Cell 1: !pip install -q transformers accelerate sentencepiece huggingface_hub
  Cell 2: from huggingface_hub import login; import os; login(token=os.environ.get("HF_TOKEN"))
  Cell 3: from google.colab import drive; drive.mount('/content/drive')
"""


import os, torch, torch.nn as nn
import json, time, gc, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Config ──────────────────────────────────────────────────────────────────
MODEL   = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
DRIVE   = "/content/drive/MyDrive/coupled_manifold_live"
RANK    = 8
HUTCH_N = 8          # Hutchinson samples per trace estimate
MAX_CTX = 128        # tokens fed to Hessian (keep small — memory)
MAX_NEW = 1028        # generation length

for d in ("sessions", "checkpoints", "logs"):
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Load model ───────────────────────────────────────────────────────────────
print(f"\nLoading {MODEL}...")
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=torch.bfloat16,
    attn_implementation="eager",   # required for create_graph=True
    device_map=None,
).to(device)
for p in model.parameters():
    p.requires_grad = False

# ── LoRA adapter (observation only — aB stays zero, no intervention yet) ─────
class DualAdapter(nn.Module):
    def __init__(self, base, rank=RANK):
        super().__init__()
        self.base = base
        inf, outf = base.in_features, base.out_features
        dev = next(base.parameters()).device
        dt  = next(base.parameters()).dtype

        self.lA = nn.Parameter(torch.randn(rank, inf,  device=dev, dtype=dt) * 0.01)
        self.lB = nn.Parameter(torch.zeros(outf, rank, device=dev, dtype=dt))

        # anti-LoRA — stays zero until intervention phase; included for continuity
        self.aA = nn.Parameter(torch.randn(rank, inf,  device=dev, dtype=dt) * 0.01)
        self.aB = nn.Parameter(torch.zeros(outf, rank, device=dev, dtype=dt))

        self.scale   = 2.0
        self.a_str   = 0.1
        self.lora_on = True
        self.anti_on = False   # OFF — observation mode

    def forward(self, x):
        out = self.base(x)
        if self.lora_on:
            out = out + (x @ self.lA.T) @ self.lB.T * self.scale
        if self.anti_on:
            out = out + (x @ self.aA.T) @ self.aB.T * self.a_str
        return out

# Inject adapters
adapter_count = 0
for name, mod in list(model.named_modules()):
    if name.split(".")[-1] in ("q_proj", "v_proj") and isinstance(mod, nn.Linear):
        parts  = name.split(".")
        parent = model
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], DualAdapter(mod))
        adapter_count += 1

# Enable grads on adapter params only
for n, p in model.named_parameters():
    if any(k in n for k in ("lA", "lB", "aA", "aB")):
        p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Adapters: {adapter_count} | Trainable: {trainable:,} / {total:,} "
      f"({100*trainable/total:.4f}%)")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Hutchinson trace estimator ───────────────────────────────────────────────
def compute_trace(input_ids):
    model.eval()
    for n, p in model.named_parameters():
        if any(k in n for k in ("lA", "lB", "aA", "aB")):
            p.requires_grad = True
    params = [p for n, p in model.named_parameters()
              if p.requires_grad and ("lA" in n or "lB" in n)]
    if not params:
        model.train()
        return 0.0
    ids = input_ids[:, :MAX_CTX].to(device)
    try:
        with torch.enable_grad():
            out = model(input_ids=ids, labels=ids)
            grads = torch.autograd.grad(out.loss, params, create_graph=True, allow_unused=True)
            grads = [g if g is not None else torch.zeros_like(p) for g, p in zip(grads, params)]
            estimates = []
            for _ in range(HUTCH_N):
                vs = [torch.randint(0, 2, p.shape, device=device, dtype=p.dtype) * 2 - 1 for p in params]
                gv = sum((g * v).sum() for g, v in zip(grads, vs))
                hvp = torch.autograd.grad(gv, params, retain_graph=True, allow_unused=True)
                hvp = [h if h is not None else torch.zeros_like(p) for h, p in zip(hvp, params)]
                estimates.append(sum((v * h).sum().item() for v, h in zip(vs, hvp)))
        torch.cuda.empty_cache()
        model.train()
        return float(np.mean(estimates))
    except RuntimeError as e:
        print(f"  [trace skipped: {e}]")
        torch.cuda.empty_cache()
        model.train()
        return 0.0

# ── Self-correcting SnobLine ──────────────────────────────────────────────────
class SnobLine:
    """
    Adaptive curvature monitor.
    Thresholds = percentiles of rolling window of own trace history.
    In observation mode: logs switches but anti-LoRA does nothing.
    """
    def __init__(self, window=8, low_pct=30, high_pct=50, max_anti=1):
        self.window   = window
        self.low_pct  = low_pct
        self.high_pct = high_pct
        self.max_anti = max_anti
        self.mode       = "lora"
        self.history    = []     # only LoRA-phase traces
        self.anti_count = 0
        self.log        = []

    def step(self, trace: float, turn: int) -> str:
        # Count anti steps regardless; force exit after max_anti
        if self.mode == "anti":
            self.anti_count += 1
            if self.anti_count >= self.max_anti:
                self.mode = "lora"
                self.anti_count = 0
                self.log.append({"turn": turn, "to": "lora", "reason": "max_duration"})
                return self.mode
        else:
            self.history.append(trace)   # only track LoRA-phase traces

        if len(self.history) < 3:
            return self.mode

        recent = self.history[-self.window:]
        low    = float(np.percentile(recent, self.low_pct))
        high   = float(np.percentile(recent, self.high_pct))

        if self.mode == "lora" and trace < low:
            self.mode = "anti"
            self.anti_count = 0
            self.log.append({"turn": turn, "to": "anti",
                             "trace": trace, "low": low, "high": high})
        elif self.mode == "anti" and trace > high:
            self.mode = "lora"
            self.anti_count = 0
            self.log.append({"turn": turn, "to": "lora",
                             "trace": trace, "reason": "recovered"})
        return self.mode

# ── Prompt builder (Qwen-based DeepSeek-R1 distill) ─────────────────────────
def build_prompt(history: list[dict]) -> str:
    """
    DeepSeek-R1-Distill-Qwen uses standard chat template.
    Falls back to manual construction if tokenizer lacks apply_chat_template.
    """
    messages = [{"role": d["role"], "content": d["content"]} for d in history]
    try:
        return tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        # Manual fallback
        out = ""
        for m in messages:
            if m["role"] == "user":
                out += f"<|im_start|>user\n{m['content']}<|im_end|>\n"
            else:
                out += f"<|im_start|>assistant\n{m['content']}<|im_end|>\n"
        out += "<|im_start|>assistant\n"
        return out

# ── set_mode helper ───────────────────────────────────────────────────────
def set_mode(m):
    for mod in model.modules():
        if isinstance(mod, DualAdapter):
            mod.lora_on = m in ("lora", "both")
            mod.anti_on = m in ("anti", "both")

# ── Gradio chat ───────────────────────────────────────────────────────────
import gradio as gr

ctrl = SnobLine()
session_id = int(time.time())
session_log = []
turn_count = [0]

css = """
.gradio-container { max-width: 850px !important; margin: auto; }
.chatbot { border-radius: 20px !important; }
.message.bot { background: rgba(76,175,80,0.06) !important; border-radius: 16px !important; }
.message.user { background: rgba(33,150,243,0.08) !important; border-radius: 16px !important; }
footer { display: none !important; }
.textbox textarea { border-radius: 12px !important; }
.markdown code { background: rgba(76,175,80,0.15) !important; color: #4CAF50 !important; border-radius: 4px; padding: 2px 6px; }
"""


opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)

def chat(user_msg, history):
    turn_count[0] += 1
    prompt = tok.apply_chat_template(
        [{"role":"user","content":user_msg}],
        tokenize=False, add_generation_prompt=True
    )
    ids = tok(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        out_ids = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=True,
                                 temperature=0.7, top_p=0.95, pad_token_id=tok.eos_token_id)
    response = tok.decode(out_ids[0][ids.shape[1]:], skip_special_tokens=True)
    if "</think>" in response:
        clean = response.split("</think>")[-1].strip()
    else:
        clean = response

    # Measure trace
    trace = compute_trace(out_ids[:, :min(out_ids.shape[1], MAX_CTX)])
    mode = ctrl.step(trace, turn_count[0])
    set_mode(mode)

    # Online learning — update adapters on this response
    with torch.enable_grad():
        learn_ids = out_ids[:, :min(out_ids.shape[1], MAX_CTX)].to(device)
        out = model(input_ids=learn_ids, labels=learn_ids)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step()
        opt.zero_grad()
    torch.cuda.empty_cache()

    icon = "🟢" if mode == "lora" else "🔴"
    bar = "█" * min(max(int(abs(trace) / 500), 1), 20)
    session_log.append({"turn":turn_count[0],"user":user_msg,"response":clean,"trace":trace,"mode":mode})
    with open(f"{DRIVE}/sessions/session_{session_id}.json","w") as f:
        json.dump({"turns":session_log,"switches":ctrl.log},f,indent=2)
    history.append({"role": "user", "content": user_msg})
    history.append({"role": "assistant", "content": clean + f"\n\n`{icon} {mode} · trace {trace:.0f} · t{turn_count[0]}` `{bar}`"})
    return "", history

with gr.Blocks(css=css, title="Coupled Manifold") as app:
    gr.Markdown("# coupled manifold\n`DeepSeek-R1 7B` · `Hessian trace` · `self-correcting`")
    chatbot = gr.Chatbot(height=550, show_label=False, container=False, type="messages")
    msg = gr.Textbox(placeholder="say something...", show_label=False, container=False)
    msg.submit(chat, [msg, chatbot], [msg, chatbot])

app.launch(share=True, debug=True)

Device: cuda
GPU: NVIDIA A100-SXM4-40GB | 42.4 GB

Loading deepseek-ai/DeepSeek-R1-Distill-Qwen-7B...


KeyboardInterrupt: 